# 03 — Custom Distortions (Plugin Registry)

The NightmareNet distortion engine is plugin-based. The Wake / Dream / Nightmare / Compress phases call into a `DistortionRegistry` that ships with two built-in engines (`dream`, `nightmare`) but accepts arbitrary registered functions matching this signature:

```python
DistortionFn = Callable[[str, float, Optional[int]], str]
```

This notebook walks through:
1. Inspecting the built-in registry
2. Writing a `DistortionFn` (homoglyph swap — a real adversarial NLP attack)
3. Registering it via a thin `@register_distortion` decorator
4. Running a multi-strength robustness sweep using the custom distortion
5. Comparing the custom distortion to `nightmare` head-to-head

**Runtime:** ~1-2 minutes on CPU. No GPU required for distortion-only work.

## 1. Install

In [ ]:
import os
import sys
import subprocess

for candidate in ['..', '.', '/content/NightmareNet']:
    if os.path.exists(os.path.join(candidate, 'pyproject.toml')):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', candidate])
        break
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nightmarenet'])

## 2. Inspect the built-in registry

Every NightmareNet process talks to the same lazy-singleton registry returned by `get_registry()`. Out of the box it ships with two engines.

In [ ]:
from nightmarenet.distortions.registry import get_registry

registry = get_registry()
for engine in registry.list_engines():
    print(engine)

## 3. Define a `@register_distortion` decorator

The registry exposes `register(name, fn, metadata)` directly. For ergonomic plugin authoring, a one-line decorator wrapping that call is conventional. The snippet below is the recommended pattern for distributing a third-party distortion package — paste it into your own `setup.py`-installable package alongside the `DistortionFn` itself.

In [ ]:
from typing import Optional

def register_distortion(name, *, phase='custom', description=''):
    """Decorator that registers a DistortionFn with the global registry."""
    def decorator(fn):
        registry = get_registry()
        registry.register(
            name,
            fn,
            metadata={'phase': phase, 'description': description},
        )
        return fn
    return decorator

## 4. Write a custom distortion: homoglyph swap

Homoglyph attacks substitute Latin characters with visually-identical Unicode characters from other scripts (Cyrillic `e` for Latin `e`, Greek `o` for Latin `o`, etc.). They are a known weakness in tokenizer-based pipelines because the BPE merges differ entirely between the swapped variants — yet the text is indistinguishable to a human reader.

We build a `DistortionFn` that randomly swaps each character with a probability proportional to `strength`.

In [ ]:
import random

HOMOGLYPHS = {
    'a': '\u0430',  # Cyrillic a
    'c': '\u0441',  # Cyrillic s -> looks like c
    'e': '\u0435',  # Cyrillic ie
    'i': '\u0456',  # Cyrillic dotted i
    'o': '\u03bf',  # Greek omicron
    'p': '\u0440',  # Cyrillic er
    'x': '\u0445',  # Cyrillic ha
    'y': '\u0443',  # Cyrillic u -> looks like y
}

@register_distortion(
    'homoglyph',
    phase='nightmare',
    description='Latin -> visually identical Cyrillic / Greek substitution',
)
def homoglyph_distortion(text: str, strength: float, seed: Optional[int] = None) -> str:
    if seed is not None:
        rng = random.Random(seed)
    else:
        rng = random
    swap_prob = max(0.0, min(1.0, float(strength)))
    out_chars = []
    for ch in text:
        replacement = HOMOGLYPHS.get(ch.lower())
        if replacement is not None and rng.random() < swap_prob:
            out_chars.append(replacement)
        else:
            out_chars.append(ch)
    return ''.join(out_chars)

registry = get_registry()
print('Engines now registered:', registry.engine_names)

## 5. Sanity check

The output should look identical to a human eye but contain non-Latin codepoints.

In [ ]:
text = 'apocalyptic adversaries operate on extremely poisonous payloads'
for s in [0.0, 0.25, 0.5, 0.75, 1.0]:
    out = registry.apply('homoglyph', text, strength=s, seed=42)
    swapped = sum(1 for ch in out if ord(ch) > 127)
    print(f'strength={s:.2f}  swapped={swapped:>3}  -> {out}')

## 6. Run a robustness sweep using the custom distortion

We now use the custom engine inside the same multi-strength evaluation loop NightmareNet uses internally. We do this without loading a real model — just measuring the *distortion characteristic curve* (token overlap with the original) — because the goal is to validate that the plugin is wired correctly, not to retrain a model.

For an end-to-end model evaluation, see `nightmarenet.evaluation.evaluator.Evaluator.evaluate(..., distortion_fn=...)`.

In [ ]:
samples = [
    'I love this movie so much.',
    'The cinematography was breathtaking.',
    'Acting was wooden and the dialogue felt forced.',
    'Climate scientists agree warming is anthropogenic.',
]
strengths = [0.1, 0.3, 0.5, 0.7, 0.9]

def token_retention(original: str, distorted: str) -> float:
    orig_tokens = set(original.split())
    dist_tokens = set(distorted.split())
    if not orig_tokens:
        return 1.0
    return len(orig_tokens & dist_tokens) / len(orig_tokens)

def sweep(engine_name):
    rows = []
    for s in strengths:
        retentions = []
        for text in samples:
            distorted = registry.apply(engine_name, text, strength=s, seed=42)
            retentions.append(token_retention(text, distorted))
        rows.append(sum(retentions) / len(retentions))
    return rows

homo = sweep('homoglyph')
nightmare = sweep('nightmare')

print(f"{'strength':>8} {'homoglyph':>12} {'nightmare':>12}")
for s, h, n in zip(strengths, homo, nightmare):
    print(f'{s:>8.2f} {h:>12.3f} {n:>12.3f}')

## 7. Visualise the distortion characteristic curve

In [ ]:
import subprocess as _sp
_sp.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'])

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(strengths, homo, marker='o', label='homoglyph (custom)', color='#F59E0B')
plt.plot(strengths, nightmare, marker='s', label='nightmare (built-in)', color='#EF4444')
plt.xlabel('Distortion strength')
plt.ylabel('Token retention vs original')
plt.title('Distortion characteristic curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. End-to-end model evaluation with the custom distortion

When you are ready to test against a real model, plug the custom function directly into `Evaluator`. The signature `(text, strength, seed) -> str` lines up with what the evaluator expects.

```python
from nightmarenet.evaluation.evaluator import Evaluator

evaluator = Evaluator(model=my_model, tokenizer=my_tokenizer, config=cfg, device='cpu')
results = evaluator.evaluate(
    clean_dataloader=test_dl,
    base_dataset=test_dataset,
    distortion_fn=homoglyph_distortion,
    label='homoglyph-eval',
)
print(results['robustness'])
```

Because the registry is process-global, the `nightmarenet train` and `nightmarenet evaluate` CLI commands also see your custom engine the moment your decorator runs.

## Next steps

- Read [`nightmarenet/distortions/registry.py`](../nightmarenet/distortions/registry.py) for the full registry API.
- Read the paper draft ([`docs/research/paper-draft.md`](../docs/research/paper-draft.md)) for the formal definition of how distortions enter the 4-phase cycle.
- See [`CONTRIBUTING.md`](../CONTRIBUTING.md) for how to upstream a new distortion engine into the core repo.
- For inspiration, browse open issues tagged `distortion-plugin-wanted` on GitHub.